# ResNet50 - PetFinder

Entrenamiento de un modelo de imágenes usando **ResNet50** preentrenado en ImageNet con transfer learning.

**Objetivo:** Predecir `AdoptionSpeed` a partir de la foto principal de cada mascota. Las predicciones se guardan para luego integrarlas con el modelo tabular.

**Flujo:**
1. Preparar datos e imágenes
2. Definir el modelo
3. Entrenar
4. Evaluar
5. Guardar predicciones

## 1. Imports y configuración

In [ ]:
import numpy as np
import pandas as pd
import os
import shutil
import time
import copy
import datetime

import torch
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, confusion_matrix

from tqdm import tqdm
from joblib import dump

# Verificamos GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando: {device} - {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# Paths
BASE_DIR       = '../'
PATH_TRAIN_CSV = os.path.join(BASE_DIR, 'input/petfinder-adoption-prediction/train/train.csv')
PATH_IMAGES    = os.path.join(BASE_DIR, 'input/petfinder-adoption-prediction/train_images')
PATH_TRAIN_DIR = os.path.join(BASE_DIR, 'work/resnet_train')
PATH_VAL_DIR   = os.path.join(BASE_DIR, 'work/resnet_val')
PATH_MODELS    = os.path.join(BASE_DIR, 'work/models')

os.makedirs(PATH_TRAIN_DIR, exist_ok=True)
os.makedirs(PATH_VAL_DIR,   exist_ok=True)
os.makedirs(PATH_MODELS,    exist_ok=True)

# Hiperparámetros
SEED        = 42
IMAGE_SIZE  = 224   # Tamaño estándar de ResNet50
BATCH_SIZE  = 32    # Ajustado para RTX 2060 (6GB VRAM)
NUM_EPOCHS  = 5
LR          = 0.001
TEST_SIZE   = 0.2
NUM_WORKERS = 0     # En Windows usar 0 para evitar errores de multiprocessing

CLASS_NAMES = ['0', '1', '2', '3', '4']

print('Configuración lista.')

## 2. Preparar datos

PyTorch espera las imágenes organizadas en carpetas por clase:
```
resnet_train/
    0/  ← imágenes de AdoptionSpeed=0
    1/  ← imágenes de AdoptionSpeed=1
    ...
resnet_val/
    0/
    ...
```

In [ ]:
train_df = pd.read_csv(PATH_TRAIN_CSV)
print(f'Total registros: {len(train_df)}')
print(f'Distribución target:\n{train_df["AdoptionSpeed"].value_counts().sort_index()}')

In [ ]:
# Split estratificado para mantener la proporción de clases en train y val
train_data, val_data = train_test_split(
    train_df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=train_df['AdoptionSpeed']
)
print(f'Train: {len(train_data)} | Val: {len(val_data)}')

In [ ]:
# Crear carpetas por clase dentro de train y val
for clase in CLASS_NAMES:
    os.makedirs(os.path.join(PATH_TRAIN_DIR, clase), exist_ok=True)
    os.makedirs(os.path.join(PATH_VAL_DIR,   clase), exist_ok=True)

def copiar_imagenes(data, directorio_destino):
    """Copia la foto principal de cada mascota a la carpeta de su clase."""
    copiadas   = 0
    sin_imagen = 0
    for _, row in data.iterrows():
        pet_id  = row['PetID']
        clase   = str(row['AdoptionSpeed'])
        origen  = os.path.join(PATH_IMAGES, f'{pet_id}-1.jpg')
        destino = os.path.join(directorio_destino, clase, f'{pet_id}-1.jpg')
        if os.path.exists(origen):
            shutil.copy2(origen, destino)
            copiadas += 1
        else:
            sin_imagen += 1
    print(f'  Copiadas: {copiadas} | Sin imagen: {sin_imagen} → {directorio_destino}')

# Solo copiar si las carpetas están vacías (evita repetir si ya se corrió)
if not any(os.scandir(os.path.join(PATH_TRAIN_DIR, '0'))):
    print('Copiando imágenes...')
    copiar_imagenes(train_data, PATH_TRAIN_DIR)
    copiar_imagenes(val_data,   PATH_VAL_DIR)
    print('Listo.')
else:
    print('Imágenes ya copiadas, saltando.')

In [ ]:
# Normalización estándar de ImageNet (requerida por ResNet preentrenado)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Para train: augmentación simple para mejorar generalización
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Para val: sin augmentación
val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

train_dataset = datasets.ImageFolder(PATH_TRAIN_DIR, transform=train_transforms)
val_dataset   = datasets.ImageFolder(PATH_VAL_DIR,   transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
print(f'Clases detectadas: {train_dataset.class_to_idx}')

## 3. Definir el modelo

Usamos **transfer learning**:
- Cargamos ResNet50 ya entrenado en ImageNet (sabe detectar bordes, texturas, formas, objetos)
- Reemplazamos la última capa que clasificaba 1000 categorías de ImageNet por una nueva que clasifica nuestras 5 clases de `AdoptionSpeed`
- Entrenamos todo el modelo con LR bajo para preservar el conocimiento previo

In [ ]:
# Cargamos ResNet50 preentrenado
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Reemplazamos la última capa fully connected
# num_features = 2048 (lo que produce ResNet antes de clasificar)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 5)  # 5 clases de AdoptionSpeed

model = model.to(device)

# Pérdida: CrossEntropyLoss (estándar para clasificación multiclase)
criterion = nn.CrossEntropyLoss()

# Optimizador SGD con momentum
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9)

# Scheduler: reduce el LR a la mitad cada 3 epochs
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

print(f'Modelo listo.')
print(f'Última capa: {model.fc}')
print(f'Parámetros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 4. Entrenamiento

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs):
    since = time.time()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_kappa     = -999
    history        = []

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}  |  LR: {scheduler.get_last_lr()[0]:.5f}')
        print('-' * 50)

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            loader   = train_loader   if phase == 'train' else val_loader
            n_samples = len(train_dataset) if phase == 'train' else len(val_dataset)

            running_loss     = 0.0
            running_corrects = 0
            all_preds  = []
            all_labels = []
            all_scores = []

            for inputs, labels in tqdm(loader, desc=phase):
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss     += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

                if phase == 'val':
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())
                    all_scores.extend(outputs.cpu().detach().numpy())

            epoch_loss = running_loss / n_samples
            epoch_acc  = running_corrects.double() / n_samples

            if phase == 'val':
                kappa = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
                print(f'  Val   Loss: {epoch_loss:.4f}  Acc: {epoch_acc*100:.2f}%  Kappa: {kappa:.4f}')
                history.append({'epoch': epoch+1, 'val_loss': epoch_loss,
                                'val_acc': float(epoch_acc), 'kappa': kappa})
                if kappa > best_kappa:
                    best_kappa     = kappa
                    best_model_wts = copy.deepcopy(model.state_dict())
                    best_scores    = all_scores
                    best_preds     = all_preds
                    best_labels    = all_labels
                    print(f'  *** Nuevo mejor kappa: {best_kappa:.4f} ***')
            else:
                print(f'  Train Loss: {epoch_loss:.4f}  Acc: {epoch_acc*100:.2f}%')

        scheduler.step()

    elapsed = time.time() - since
    print(f'\nEntrenamiento finalizado en {elapsed//60:.0f}m {elapsed%60:.0f}s')
    print(f'Mejor Kappa: {best_kappa:.4f}')

    model.load_state_dict(best_model_wts)
    return model, best_kappa, best_scores, best_preds, best_labels, pd.DataFrame(history)

In [ ]:
model, best_kappa, val_scores, val_preds, val_labels, history = train_model(
    model, criterion, optimizer, scheduler, num_epochs=NUM_EPOCHS
)

## 5. Evaluación

In [ ]:
# Curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history['epoch'], history['val_loss'], marker='o', color='steelblue')
axes[0].set_title('Val Loss por epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(history['epoch'], history['kappa'], marker='o', color='salmon')
axes[1].set_title('Kappa cuadrático por epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Kappa')

plt.tight_layout()
plt.show()

print(history)

In [ ]:
# Matriz de confusión del mejor modelo
cm = confusion_matrix(val_labels, val_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

plt.figure(figsize=(8, 6))
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Matriz de confusión (%) — Kappa: {best_kappa:.4f}')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

## 6. Guardar modelo y predicciones

Guardamos:
- Los **pesos del modelo** para reutilizarlo sin reentrenar
- Las **predicciones sobre val** (scores crudos de 5 clases por mascota) para integrarlas al modelo tabular

In [ ]:
run_id = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Guardar pesos del modelo
model_path = os.path.join(PATH_MODELS, f'resnet50_petfinder_{run_id}.pth')
torch.save(model.state_dict(), model_path)
print(f'Modelo guardado en: {model_path}')

# Armar DataFrame de predicciones con PetID
val_pet_ids = [
    s[0].replace('\\', '/').split('/')[-1].split('-')[0]
    for s in val_loader.dataset.samples
]

pred_df = pd.DataFrame({
    'PetID':             val_pet_ids,
    'resnet_pred_score': val_scores,   # vector de 5 scores por mascota
    'resnet_pred':       val_preds,    # clase predicha
    'AdoptionSpeed':     val_labels    # clase real
})

# Guardar predicciones
pred_path = os.path.join(PATH_MODELS, f'resnet50_val_preds_{run_id}.joblib')
dump(pred_df, pred_path)
print(f'Predicciones guardadas en: {pred_path}')

pred_df.head()

## 7. Próximos pasos

Con las predicciones guardadas podés:

- **Integrar con el modelo tabular**: sumar los scores del ResNet con los del modelo tabular (blend) para aprovechar ambas señales.
- **Fine-tuning**: descongelar solo las últimas capas de ResNet y reentrenar con LR muy bajo.
- **Más augmentación**: rotaciones, zoom, cutout para mejorar la generalización.
- **Múltiples fotos**: promediar predicciones de todas las fotos de cada mascota en lugar de usar solo la primera.